In [ ]:
import os
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import accuracy_score, roc_auc_score, precision_score, recall_score, f1_score
from sklearn.metrics import matthews_corrcoef, roc_curve, auc, confusion_matrix
from sklearn.preprocessing import label_binarize, StandardScaler
from sklearn.feature_selection import SelectFromModel
from sklearn.linear_model import Lasso
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt
from matplotlib import rcParams
import seaborn as sns
from tqdm import tqdm
import shap
from mrmr import mrmr_classif

In [ ]:
# Set device
device = torch.device("mps" if torch.backends.mps.is_available() else "cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Model name - using ConvNext as specified
model_name = 'convnext'

# Paths
OUTPUT_PATH = f'1_Feature_Extraction/{model_name}'
RESULTS_PATH = os.path.join(f'3_Feature_Selection')
os.makedirs(RESULTS_PATH, exist_ok=True)

# Number of features to select
N_SELECTED_FEATURES = 800

In [ ]:
# Load feature data
try:
    train_df = pd.read_csv(os.path.join(OUTPUT_PATH, f'train_features_{model_name}.csv'))
    test_df = pd.read_csv(os.path.join(OUTPUT_PATH, f'test_features_{model_name}.csv'))
    print(f"Loaded {len(train_df)} training and {len(test_df)} testing samples")
except FileNotFoundError:
    print(f"ConvNext features not found, attempting to use available features")
    train_df = pd.read_csv(os.path.join('1_Feature_Extraction/densenet121', f'train_features_densenet121.csv'))
    test_df = pd.read_csv(os.path.join('1_Feature_Extraction/densenet121', f'test_features_densenet121.csv'))
    print(f"Using DenseNet features with {len(train_df)} training and {len(test_df)} testing samples")

In [ ]:
# Get the actual feature dimension
feature_columns = [col for col in train_df.columns if col.startswith('feat_')]
feature_dim = len(feature_columns)
print(f"Total number of features: {feature_dim}")

# Extract features and labels with 4 classes
X_train = train_df[feature_columns].values
y_train = train_df['label'].map({'glioma': 0, 'meningioma': 1, 'notumor': 2, 'pituitary': 3}).values
X_test = test_df[feature_columns].values
y_test = test_df['label'].map({'glioma': 0, 'meningioma': 1, 'notumor': 2, 'pituitary': 3}).values

# Define the 4 classes
CLASSES = ['glioma', 'meningioma', 'notumor', 'pituitary']
N_CLASSES = len(CLASSES)  # Should be 4
print(f"Number of classes: {N_CLASSES}")

In [ ]:
# Plot settings
rcParams['font.family'] = 'Times New Roman'
rcParams['axes.titlesize'] = 28
rcParams['axes.titlepad'] = 20
rcParams['axes.labelsize'] = 23
rcParams['xtick.labelsize'] = 18
rcParams['ytick.labelsize'] = 18
rcParams['legend.fontsize'] = 16
rcParams['lines.linewidth'] = 3
rcParams['axes.linewidth'] = 2

In [ ]:
# ===================== FEATURE SELECTION METHODS ===================== #

def apply_pca(X_train, X_test, n_components=N_SELECTED_FEATURES):
    print(f"Applying PCA to select {n_components} components...")
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)
    
    pca = PCA(n_components=n_components)
    X_train_pca = pca.fit_transform(X_train_scaled)
    X_test_pca = pca.transform(X_test_scaled)
    
    explained_var = np.sum(pca.explained_variance_ratio_) * 100
    print(f"Explained variance with {n_components} components: {explained_var:.2f}%")
    
    plt.figure(figsize=(10, 6))
    plt.plot(np.cumsum(pca.explained_variance_ratio_), marker='o', linestyle='--')
    plt.xlabel('Number of Components')
    plt.ylabel('Cumulative Explained Variance')
    plt.title('PCA Explained Variance')
    plt.grid(True)
    plt.savefig(os.path.join(RESULTS_PATH, 'pca_explained_variance.png'), dpi=1000, bbox_inches='tight')
    plt.savefig(os.path.join(RESULTS_PATH, 'pca_explained_variance.pdf'), dpi=1000, bbox_inches='tight')
    plt.close()
    
    return X_train_pca, X_test_pca, pca

In [ ]:
def apply_lasso(X_train, X_test, n_features=N_SELECTED_FEATURES):
    print(f"Applying Lasso to select {n_features} features...")
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)
    
    from sklearn.linear_model import LassoCV
    lasso_cv = LassoCV(cv=5, random_state=42, max_iter=10000)
    lasso_cv.fit(X_train_scaled, y_train)
    print(f"Optimal alpha: {lasso_cv.alpha_:.6f}")
    
    lasso = Lasso(alpha=lasso_cv.alpha_, max_iter=10000)
    lasso.fit(X_train_scaled, y_train)
    
    feature_importance = np.abs(lasso.coef_)
    selected_indices = np.argsort(feature_importance)[::-1][:n_features]
    
    selected_indices = np.array(selected_indices, dtype=int)
    
    plt.figure(figsize=(10, 6))
    plt.bar(range(20), feature_importance[selected_indices[:20]])
    plt.xlabel('Feature Index')
    plt.ylabel('Importance Score')
    plt.title('Top 20 Lasso Feature Importance')
    plt.grid(True)
    plt.savefig(os.path.join(RESULTS_PATH, 'lasso_feature_importance.png'), dpi=1000, bbox_inches='tight')
    plt.savefig(os.path.join(RESULTS_PATH, 'lasso_feature_importance.pdf'), dpi=1000, bbox_inches='tight')
    plt.close()
    
    X_train_lasso = X_train[:, selected_indices]
    X_test_lasso = X_test[:, selected_indices]
    
    return X_train_lasso, X_test_lasso, selected_indices

In [ ]:
def apply_mrmr(X_train, X_test, n_features=N_SELECTED_FEATURES):
    print(f"Applying mRMR to select {n_features} features...")
    feature_names = [f"feat_{i}" for i in range(X_train.shape[1])]
    X_train_df = pd.DataFrame(X_train, columns=feature_names)
    
    selected_features = mrmr_classif(X_train_df, pd.Series(y_train), K=n_features)
    print(f"Selected {len(selected_features)} features using mRMR")
    
    selected_indices = np.array([int(feat.split('_')[1]) for feat in selected_features], dtype=int)
    
    plt.figure(figsize=(12, 8))
    plt.barh(range(20), [1/(i+1) for i in range(20)], align='center')
    plt.yticks(range(20), selected_features[:20])
    plt.xlabel('Relative Importance')
    plt.ylabel('Feature')
    plt.title('Top 20 mRMR Selected Features')
    plt.tight_layout()
    plt.savefig(os.path.join(RESULTS_PATH, 'mrmr_selected_features.png'), dpi=1000, bbox_inches='tight')
    plt.savefig(os.path.join(RESULTS_PATH, 'mrmr_selected_features.pdf'), dpi=1000, bbox_inches='tight')
    plt.close()
    
    X_train_mrmr = X_train[:, selected_indices]
    X_test_mrmr = X_test[:, selected_indices]
    
    return X_train_mrmr, X_test_mrmr, selected_indices

In [ ]:
def apply_shap_multiclass(X_train, X_test, n_features=N_SELECTED_FEATURES):
    print(f"Applying SHAP to select {n_features} features for 4-class problem...")
    from sklearn.ensemble import RandomForestClassifier
    
    # Train a model for SHAP analysis - use a smaller model for speed
    print("Training RandomForest model for SHAP analysis...")
    rf_model = RandomForestClassifier(n_estimators=50, max_depth=10, random_state=42)
    rf_model.fit(X_train, y_train)
    
    # Use a smaller sample for efficiency
    n_samples = min(300, X_train.shape[0])
    X_train_sample = X_train[:n_samples]
    print(f"Using {n_samples} samples for SHAP analysis")
    
    try:
        # Alternative approach using feature importance directly
        print("Using RandomForest feature importance as base...")
        feature_importance = rf_model.feature_importances_
        
        # Select top features based on RandomForest importance
        selected_indices = np.argsort(feature_importance)[::-1][:n_features]
        selected_indices = np.array(selected_indices, dtype=int)
        
        # Create feature names array
        feature_names = [f"Feature {i}" for i in range(X_train.shape[1])]
        selected_feature_names = [feature_names[i] for i in selected_indices]
        
        # Plot the feature importance
        plt.figure(figsize=(12, 8))
        top_20_indices = selected_indices[:20]
        top_20_importances = feature_importance[top_20_indices]
        top_20_names = [f"F{i}" for i in top_20_indices]
        
        plt.barh(range(20), top_20_importances[::-1], align='center')
        plt.yticks(range(20), top_20_names[::-1])
        plt.xlabel('Feature Importance')
        plt.title('Top 20 Features by Importance')
        plt.tight_layout()
        plt.savefig(os.path.join(RESULTS_PATH, 'feature_importance_top20.png'), dpi=1000, bbox_inches='tight')
        plt.savefig(os.path.join(RESULTS_PATH, 'feature_importance_top20.pdf'), dpi=1000, bbox_inches='tight')
        plt.close()
        
        # Try to compute SHAP values for visualization only (not for selection)
        print("Computing SHAP values for top 20 features only...")
        try:
            # Create a new explainer with just the top 20 features for visualization
            X_top20 = X_train_sample[:, top_20_indices]
            
            # Train a simpler model just on these features
            simple_model = RandomForestClassifier(n_estimators=50, max_depth=5, random_state=42)
            simple_model.fit(X_train[:, top_20_indices], y_train)
            
            # Create an explainer
            explainer = shap.TreeExplainer(simple_model)
            
            # Get SHAP values just for this subset
            shap_values = explainer.shap_values(X_top20)
            
            # Print the structure of shap_values for debugging
            if isinstance(shap_values, list):
                print(f"SHAP values is a list of length {len(shap_values)}")
                for i, sv in enumerate(shap_values):
                    print(f"  - Shape of shap_values[{i}]: {sv.shape}")
            else:
                print(f"SHAP values is an array of shape {shap_values.shape}")
            
            # Try different approaches to create a summary plot
            plt.figure(figsize=(12, 10))
            
            try:
                # Try the simplest approach first - just the first class
                if isinstance(shap_values, list) and len(shap_values) >= 1:
                    print("Creating SHAP summary plot for the first class...")
                    shap.summary_plot(
                        shap_values[0], X_top20,
                        feature_names=[f"F{i}" for i in top_20_indices],
                        show=False
                    )
                else:
                    print("Creating simple SHAP summary plot...")
                    shap.summary_plot(
                        shap_values, X_top20,
                        feature_names=[f"F{i}" for i in top_20_indices],
                        show=False
                    )
                
                plt.tight_layout()
                plt.savefig(os.path.join(RESULTS_PATH, 'shap_summary_plot.png'), dpi=1000, bbox_inches='tight')
                plt.savefig(os.path.join(RESULTS_PATH, 'shap_summary_plot.pdf'), dpi=1000, bbox_inches='tight')
                plt.close()
                print("Successfully created SHAP summary plot!")
                
            except Exception as e:
                print(f"Could not create SHAP summary plot: {str(e)}")
        
        except Exception as e:
            print(f"Error computing SHAP values for visualization: {str(e)}")
        
        # Return the selected features based on RandomForest importance
        X_train_selected = X_train[:, selected_indices]
        X_test_selected = X_test[:, selected_indices]
        
        return X_train_selected, X_test_selected, selected_indices
        
    except Exception as e:
        print(f"Error in SHAP/RF feature selection: {str(e)}")
        
        # Last resort: fall back to PCA
        print("Falling back to PCA for feature selection")
        scaler = StandardScaler()
        X_train_scaled = scaler.fit_transform(X_train)
        X_test_scaled = scaler.transform(X_test)
        
        pca = PCA(n_components=n_features)
        X_train_pca = pca.fit_transform(X_train_scaled)
        X_test_pca = pca.transform(X_test_scaled)
        
        return X_train_pca, X_test_pca, None

In [ ]:
# ===================== MODEL DEFINITION ===================== #

class FeatureDataset(Dataset):
    def __init__(self, features, labels):
        self.features = torch.tensor(features, dtype=torch.float32)
        self.labels = torch.tensor(labels, dtype=torch.long)
    
    def __len__(self):
        return len(self.labels)
    
    def __getitem__(self, idx):
        return self.features[idx], self.labels[idx]

In [ ]:
class Attention(nn.Module):
    def __init__(self, feature_dim):
        super(Attention, self).__init__()
        self.attention = nn.Sequential(
            nn.Linear(feature_dim, feature_dim // 2),
            nn.Tanh(),
            nn.Linear(feature_dim // 2, 1),
            nn.Softmax(dim=1)
        )
    
    def forward(self, x):
        weights = self.attention(x)
        return (x * weights).sum(dim=1)

In [ ]:
class AttGRU(nn.Module):
    def __init__(self, input_dim, hidden_dim=512, num_classes=N_CLASSES):  # 4 classes
        super(AttGRU, self).__init__()
        self.gru = nn.GRU(input_dim, hidden_dim, batch_first=True)
        self.attention = Attention(hidden_dim)
        self.fc = nn.Linear(hidden_dim, num_classes)
    
    def forward(self, x):
        x = x.unsqueeze(1)  # Add sequence dimension
        x, _ = self.gru(x)
        x = self.attention(x)
        x = self.fc(x)
        return x

In [ ]:
def train_model(model, train_loader, test_loader, feature_method, num_epochs=50, lr=0.001):
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    train_losses = []
    test_losses = []
    train_accs = []
    test_accs = []
    
    checkpoint_path = os.path.join(RESULTS_PATH, f'AttGRU_{feature_method}_checkpoint.pth')
    
    for epoch in tqdm(range(num_epochs), desc=f"Training AttGRU with {feature_method}"):
        model.train()
        running_loss = 0.0
        correct = 0
        total = 0
        
        for inputs, labels in train_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            running_loss += loss.item()
            
            _, predicted = outputs.max(1)
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()
        
        train_loss = running_loss / len(train_loader)
        train_acc = correct / total
        train_losses.append(train_loss)
        train_accs.append(train_acc)
        
        model.eval()
        test_loss = 0.0
        correct = 0
        total = 0
        with torch.no_grad():
            for inputs, labels in test_loader:
                inputs, labels = inputs.to(device), labels.to(device)
                outputs = model(inputs)
                loss = criterion(outputs, labels)
                test_loss += loss.item()
                
                _, predicted = outputs.max(1)
                total += labels.size(0)
                correct += predicted.eq(labels).sum().item()
        
        test_loss = test_loss / len(test_loader)
        test_acc = correct / total
        test_losses.append(test_loss)
        test_accs.append(test_acc)
    
    torch.save(model.state_dict(), checkpoint_path)
    print(f"Saved AttGRU_{feature_method} checkpoint to {checkpoint_path}")
    
    return train_losses, test_losses, train_accs, test_accs

In [ ]:
def evaluate_model(model, X_test, y_test):
    model.eval()
    with torch.no_grad():
        X_test_tensor = torch.tensor(X_test, dtype=torch.float32).to(device)
        outputs = model(X_test_tensor)
        _, y_pred = outputs.max(1)
        y_pred = y_pred.cpu().numpy()
        y_prob = torch.softmax(outputs, dim=1).cpu().numpy()
    
    metrics = {}
    metrics['ACC'] = accuracy_score(y_test, y_pred)
    metrics['AUC'] = roc_auc_score(y_test, y_prob, multi_class='ovr')
    metrics['PRE'] = precision_score(y_test, y_pred, average='macro')
    metrics['SN'] = recall_score(y_test, y_pred, average='macro')
    metrics['F1'] = f1_score(y_test, y_pred, average='macro')
    metrics['MCC'] = matthews_corrcoef(y_test, y_pred)
    
    return metrics, y_prob, y_pred

In [ ]:
def plot_curves(train_losses, test_losses, train_accs, test_accs, feature_method):
    plt.figure(figsize=(12, 5))
    
    plt.subplot(1, 2, 1)
    plt.plot(train_losses, label='Train Loss')
    plt.plot(test_losses, label='Test Loss')
    plt.title(f'AttGRU with {feature_method} - Loss Curves')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.legend()
    plt.grid(True)
    
    plt.subplot(1, 2, 2)
    plt.plot(train_accs, label='Train Accuracy')
    plt.plot(test_accs, label='Test Accuracy')
    plt.title(f'AttGRU with {feature_method} - Accuracy Curves')
    plt.xlabel('Epoch')
    plt.ylabel('Accuracy')
    plt.legend()
    plt.grid(True)
    
    plt.tight_layout()
    plt.savefig(os.path.join(RESULTS_PATH, f'AttGRU_{feature_method}_learning_curves.png'), dpi=1000, bbox_inches='tight')
    plt.savefig(os.path.join(RESULTS_PATH, f'AttGRU_{feature_method}_learning_curves.pdf'), dpi=1000, bbox_inches='tight')
    plt.close()

In [ ]:
def plot_roc(y_test, y_prob, feature_method):
    y_test_bin = label_binarize(y_test, classes=range(N_CLASSES))  # 4 classes
    fpr, tpr, roc_auc = {}, {}, {}
    
    plt.figure(figsize=(8, 8))
    for i in range(N_CLASSES):  # Loop over 4 classes
        fpr[i], tpr[i], _ = roc_curve(y_test_bin[:, i], y_prob[:, i])
        roc_auc[i] = auc(fpr[i], tpr[i])
        plt.plot(fpr[i], tpr[i], label=f'{CLASSES[i]} (AUC = {roc_auc[i]:.2f})')
    
    plt.plot([0, 1], [0, 1], 'k--')
    plt.title(f'AttGRU with {feature_method} - ROC Curve')
    plt.xlabel('False Positive Rate')
    plt.ylabel('True Positive Rate')
    plt.legend(loc='lower right')
    plt.grid(True)
    plt.savefig(os.path.join(RESULTS_PATH, f'AttGRU_{feature_method}_roc_curve.png'), dpi=1000, bbox_inches='tight')
    plt.savefig(os.path.join(RESULTS_PATH, f'AttGRU_{feature_method}_roc_curve.pdf'), dpi=1000, bbox_inches='tight')
    plt.close()

In [ ]:
def plot_confusion_matrix(y_test, y_pred, feature_method):
    cm = confusion_matrix(y_test, y_pred)
    plt.figure(figsize=(8, 8))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=CLASSES, yticklabels=CLASSES,
                annot_kws={"size": 18})
    plt.title(f'AttGRU with {feature_method} - Confusion Matrix')
    plt.xlabel('Predicted')
    plt.ylabel('True')
    plt.savefig(os.path.join(RESULTS_PATH, f'AttGRU_{feature_method}_confusion_matrix.png'), dpi=1000, bbox_inches='tight')
    plt.savefig(os.path.join(RESULTS_PATH, f'AttGRU_{feature_method}_confusion_matrix.pdf'), dpi=1000, bbox_inches='tight')
    plt.close()

In [ ]:
# ===================== ALTERNATIVE SHAP METHOD ===================== #

def apply_shap_alternative(X_train, X_test, n_features=N_SELECTED_FEATURES):
    """Alternative implementation if the regular SHAP approach fails"""
    print(f"Applying alternative SHAP approach to select {n_features} features...")
    from sklearn.ensemble import RandomForestClassifier
    
    # Train a simple model for feature importance
    rf_model = RandomForestClassifier(n_estimators=100, random_state=42)
    rf_model.fit(X_train, y_train)
    
    # Get feature importances directly from the model
    feature_importance = rf_model.feature_importances_
    
    # Select the top features
    selected_indices = np.argsort(feature_importance)[::-1][:n_features]
    selected_indices = np.array(selected_indices, dtype=int)
    
    # Plot feature importance
    plt.figure(figsize=(10, 6))
    plt.bar(range(20), feature_importance[selected_indices[:20]])
    plt.xlabel('Feature Index')
    plt.ylabel('Importance Score')
    plt.title('Top 20 Random Forest Feature Importance')
    plt.grid(True)
    plt.savefig(os.path.join(RESULTS_PATH, 'rf_feature_importance.png'), dpi=1000, bbox_inches='tight')
    plt.savefig(os.path.join(RESULTS_PATH, 'rf_feature_importance.pdf'), dpi=1000, bbox_inches='tight')
    plt.close()
    
    # Select the features
    X_train_selected = X_train[:, selected_indices]
    X_test_selected = X_test[:, selected_indices]
    
    return X_train_selected, X_test_selected, selected_indices

In [ ]:
print(f"Starting feature selection and AttGRU model training...")
print(f"Original data shape: {X_train.shape}, {X_test.shape}")

comparison_results = []

# Apply feature selection methods
X_train_shap, X_test_shap, shap_indices = apply_shap_multiclass(X_train, X_test)

In [ ]:
X_train_pca, X_test_pca, pca_model = apply_pca(X_train, X_test)
print(f"PCA features shape: {X_train_pca.shape}, {X_test_pca.shape}")

In [ ]:
X_train_lasso, X_test_lasso, lasso_indices = apply_lasso(X_train, X_test)
print(f"Lasso features shape: {X_train_lasso.shape}, {X_test_lasso.shape}")

In [ ]:
X_train_mrmr, X_test_mrmr, mrmr_indices = apply_mrmr(X_train, X_test)
print(f"mRMR features shape: {X_train_mrmr.shape}, {X_test_mrmr.shape}")

In [ ]:
# Train and evaluate AttGRU model
batch_size = 64
num_epochs = 50

feature_sets = {
    'SHAP': (X_train_shap, X_test_shap),
    'PCA': (X_train_pca, X_test_pca),
    'Lasso': (X_train_lasso, X_test_lasso),
    'mRMR': (X_train_mrmr, X_test_mrmr),
}

In [ ]:
for method, (X_train_selected, X_test_selected) in feature_sets.items():
    print(f"\n===== Training AttGRU with {method} features =====")
    
    train_dataset = FeatureDataset(X_train_selected, y_train)
    test_dataset = FeatureDataset(X_test_selected, y_test)
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)
    
    model = AttGRU(input_dim=X_train_selected.shape[1]).to(device)  # 4 classes by default
    train_losses, test_losses, train_accs, test_accs = train_model(
        model, train_loader, test_loader, method, num_epochs=num_epochs
    )
    
    plot_curves(train_losses, test_losses, train_accs, test_accs, method)
    
    metrics, y_prob, y_pred = evaluate_model(model, X_test_selected, y_test)
    
    plot_roc(y_test, y_prob, method)
    plot_confusion_matrix(y_test, y_pred, method)
    
    comparison_results.append({
        'Feature_Method': method,
        'Input_Dim': X_train_selected.shape[1],
        'ACC': metrics['ACC'],
        'AUC': metrics['AUC'],
        'PRE': metrics['PRE'],
        'SN': metrics['SN'],
        'F1': metrics['F1'],
        'MCC': metrics['MCC']
    })
    
    print(f"AttGRU with {method} features - Performance Metrics:")
    for metric, value in metrics.items():
        print(f"{metric}: {value:.4f}")

In [ ]:
comparison_df = pd.DataFrame(comparison_results)
comparison_df.to_csv(os.path.join(RESULTS_PATH, 'AttGRU_feature_selection_comparison.csv'), index=False)
print(f"Saved comparison results to {os.path.join(RESULTS_PATH, 'AttGRU_feature_selection_comparison.csv')}")

In [ ]:
plt.figure(figsize=(12, 8))
metrics_to_plot = ['ACC', 'AUC', 'F1', 'MCC']
x = np.arange(len(feature_sets))
width = 0.2

for i, metric in enumerate(metrics_to_plot):
    values = [result[metric] for result in comparison_results]
    plt.bar(x + i*width, values, width, label=metric)

plt.xlabel('Feature Selection Method')
plt.ylabel('Score')
plt.title('AttGRU Performance with Different Feature Selection Methods')
plt.xticks(x + width * (len(metrics_to_plot)-1)/2, list(feature_sets.keys()))
plt.legend()
plt.grid(True, axis='y')
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_PATH, 'AttGRU_feature_selection_comparison.png'), dpi=1000, bbox_inches='tight')
plt.savefig(os.path.join(RESULTS_PATH, 'AttGRU_feature_selection_comparison.pdf'), dpi=1000, bbox_inches='tight')

print("Feature selection and AttGRU model evaluation complete!")